<a href="https://colab.research.google.com/github/SrijaMadarapu8/AI-ML/blob/main/rag_mcp_agent_local.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Pipeline Wrapped as an MCP Tool, Called by an Agent

Builds a small document understanding RAG pipeline (chunking, embeddings,
ChromaDB retrieval), exposes it as a single MCP tool (`search_documents`),
and gives an agent a system prompt instructing it to call that tool only
when the question needs document context otherwise it answers directly.

This version runs **entirely locally**:
- Embeddings: `sentence-transformers/all-MiniLM-L6-v2` (downloads once, then cached)
- LLM: `Qwen/Qwen2.5-1.5B-Instruct` (small instruct model with tool-calling support)


In [4]:
!pip install mcp langchain langchain-huggingface langchain-mcp-adapters langgraph chromadb sentence-transformers transformers accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60

## Sample documents
Standing in for real uploaded documents swap this for your own text
(e.g., resume content, project notes, a manual) to make the demo specific
to your own use case.

In [5]:
docs = [
    {
        "id": "doc1",
        "text": "The robotic arm project trained a UFactory X-Arm 7 to perform "
                "pick-and-place using PPO reinforcement learning inside a MuJoCo "
                "digital twin. Training was parallelized across 512+ simulated "
                "environments on AMD ROCm hardware, using domain randomization "
                "for sim-to-real transfer."
    },
    {
        "id": "doc2",
        "text": "The computer vision project implemented RANSAC, SIFT, and "
                "Structure-from-Motion for 3D reconstruction, plus Gaussian "
                "Splatting and NeRF for photorealistic rendering. It reached "
                "95% feature-matching accuracy and reconstructed scenes 40% faster."
    },
    {
        "id": "doc3",
        "text": "The RAG document-understanding pipeline chunks technical "
                "documents, embeds them with a local sentence-transformers "
                "model, stores them in a persistent ChromaDB collection, and "
                "retrieves the top matching chunks above a similarity threshold "
                "before generating a grounded answer."
    },
]
print(f"Loaded {len(docs)} sample documents.")

Loaded 3 sample documents.


## Build the vector store (local embeddings + ChromaDB)

No API key needed the embedding model downloads from Hugging Face once
and runs locally from then on.

In [6]:
import chromadb
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"

embed_model = SentenceTransformer(EMBED_MODEL_NAME)
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Recreate the collection with cosine distance explicitly — Chroma defaults
# to squared L2, which breaks the "similarity = 1 - distance" math below.
try:
    chroma_client.delete_collection("project_docs")
except Exception:
    pass  # collection may not exist yet on a fresh run

collection = chroma_client.get_or_create_collection(
    "project_docs",
    metadata={"hnsw:space": "cosine"}
)

def embed(text: str):
    return embed_model.encode(text, normalize_embeddings=True).tolist()

for doc in docs:
    collection.add(
        ids=[doc["id"]],
        embeddings=[embed(doc["text"])],
        documents=[doc["text"]],
    )

print(f"Collection now has {collection.count()} documents.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Collection now has 3 documents.


## Retrieval function (with a similarity threshold)

Note: with `normalize_embeddings=True` above, ChromaDB's default distance
metric is still Euclidean unless the collection is created with
`metadata={"hnsw:space": "cosine"}`. This threshold logic still works as a
relative filter, but if you want true cosine similarity, recreate the
collection with that metadata set.

In [7]:
def search_documents(query: str, top_k: int = 3, threshold: float = 0.3) -> str:
    query_embedding = embed(query)
    results = collection.query(query_embeddings=[query_embedding], n_results=top_k)

    matches = []
    for doc_text, distance in zip(results["documents"][0], results["distances"][0]):
        similarity = 1 - distance  # crude distance-to-similarity conversion
        if similarity >= threshold:
            matches.append(doc_text)

    if not matches:
        return "No relevant information found in the indexed documents."
    return "\n---\n".join(matches)

print(search_documents("how was the robotic arm trained?"))

The robotic arm project trained a UFactory X-Arm 7 to perform pick-and-place using PPO reinforcement learning inside a MuJoCo digital twin. Training was parallelized across 512+ simulated environments on AMD ROCm hardware, using domain randomization for sim-to-real transfer.


## Wrap retrieval as an MCP server tool

This spawns as its own subprocess, so it loads its own copy of the
embedding model (a few seconds of startup the first time it's called).

In [8]:
%%writefile rag_server.py
from mcp.server.fastmcp import FastMCP
import chromadb
from sentence_transformers import SentenceTransformer

mcp = FastMCP("rag-tools")

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection("project_docs")

def embed(text: str):
    return embed_model.encode(text, normalize_embeddings=True).tolist()

@mcp.tool()
def search_documents(query: str) -> str:
    """Retrieve information from the indexed project documents (robotic arm,
    computer vision, RAG pipeline projects). ALWAYS call this tool first for
    any question about a specific project's implementation, techniques, or
    results — never answer those from memory."""
    query_embedding = embed(query)
    results = collection.query(query_embeddings=[query_embedding], n_results=3)

    matches = []
    for doc_text, distance in zip(results["documents"][0], results["distances"][0]):
        similarity = 1 - distance
        if similarity >= 0.3:
            matches.append(doc_text)

    if not matches:
        return "No relevant information found in the indexed documents."
    return "\n---\n".join(matches)

if __name__ == "__main__":
    mcp.run()

Writing rag_server.py


## Connect an agent to the RAG tool

In [9]:
from contextlib import asynccontextmanager
import asyncio
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain.agents import create_agent
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

LLM_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
hf_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)
gen_pipeline = pipeline(
    "text-generation",
    model=hf_model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False,
    return_full_text=False,  # strip the input prompt from the output text
)
llm = HuggingFacePipeline(pipeline=gen_pipeline)
model = ChatHuggingFace(llm=llm)

SYSTEM_PROMPT_WITH_CONTEXT = (
    "You are a helpful assistant. Relevant context from the indexed project "
    "documents has already been retrieved and is included below the "
    "question. Use it to answer accurately. Do not call any tool — the "
    "context you need has already been provided."
)

SYSTEM_PROMPT_NO_CONTEXT = (
    "You are an assistant with access to a tool called `search_documents`, "
    "which retrieves information from a specific set of indexed project "
    "documents (robotic arm, computer vision, and RAG pipeline projects).\n\n"
    "Rules:\n"
    "- If the question asks about any specific project, its implementation, "
    "results, metrics, or details, call `search_documents` before "
    "answering. Do not guess project-specific details from memory.\n"
    "- If the question is general knowledge unrelated to these projects "
    "(e.g. geography, math, common facts), answer directly without calling "
    "any tool."
)
log_file = open("rag_server_stderr.log", "w")
server_params = StdioServerParameters(command="python", args=["rag_server.py"])

@asynccontextmanager
async def mcp_session():
    async with stdio_client(server_params, errlog=log_file) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            yield session

DOC_KEYWORDS = ["project", "robotic arm", "computer vision", "rag pipeline", "reconstruction", "trained", "training", "accuracy"]

async def run_rag_agent(query: str):
    needs_docs = any(kw in query.lower() for kw in DOC_KEYWORDS)

    async with mcp_session() as session:
        tools = await load_mcp_tools(session)

        if needs_docs:
            # Deterministic fast-path: retrieve directly, no tool call needed.
            search_tool = next(t for t in tools if t.name == "search_documents")
            context = await search_tool.ainvoke({"query": query})
            prompt = f"Using this context:\n{context}\n\nAnswer the question: {query}"
            agent = create_agent(model, tools, system_prompt=SYSTEM_PROMPT_WITH_CONTEXT)
        else:
            # Keyword router found no match — fall back to letting the LLM's
            # own judgment decide whether to call the tool (e.g. paraphrased
            # document questions the keyword list missed).
            prompt = query
            agent = create_agent(model, tools, system_prompt=SYSTEM_PROMPT_NO_CONTEXT)

        return await asyncio.to_thread(agent.invoke, {"messages": prompt})


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## Test queries

In [15]:
query = "What accuracy did the computer vision reconstruction project reach?"
agent_response = await run_rag_agent(query)
for m in agent_response['messages']:
    m.pretty_print()

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


================================ Human Message =================================

Using this context:
[{'type': 'text', 'text': 'The computer vision project implemented RANSAC, SIFT, and Structure-from-Motion for 3D reconstruction, plus Gaussian Splatting and NeRF for photorealistic rendering. It reached 95% feature-matching accuracy and reconstructed scenes 40% faster.', 'id': 'lc_9f3a642c-6f6b-4085-913d-9b2c430246ba'}]

Answer the question: What accuracy did the computer vision reconstruction project reach?
================================== Ai Message ==================================

The computer vision reconstruction project achieved an accuracy of 95% in feature matching.


In [12]:
query = "What is the capital of France?"
agent_response = await run_rag_agent(query)
for m in agent_response['messages']:
    m.pretty_print()

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


================================ Human Message =================================

What is the capital of France?
================================== Ai Message ==================================

The capital of France is Paris.


## Log the agent's tool-use decisions


In [16]:
needs_docs = any(kw in query.lower() for kw in DOC_KEYWORDS)
print(f"Router decision: {'deterministic retrieval (no tool call expected)' if needs_docs else 'no keyword match — LLM had the tool available'}")

for m in agent_response['messages']:
    if hasattr(m, 'tool_calls') and m.tool_calls:
        print("Agent chose to call:", [tc['name'] for tc in m.tool_calls])
    elif m.type == 'ai' and m.content:
        print("Agent answered directly without a tool call.")

Router decision: deterministic retrieval (no tool call expected)
Agent answered directly without a tool call.
